## Imports

In [ ]:
import os
import bdsf
import numpy as np
from pathlib import Path
from astropy.io import fits
from PIL import Image

## FITS → PNG

In [ ]:
def fits_to_png(input_fits_path: str, output_png_path: str):
    data = fits.getdata(input_fits_path)
    header = fits.getheader(input_fits_path)

    width, height = header["NAXIS1"], header["NAXIS2"]
    data = np.reshape(data, (height, width))

    data[np.isnan(data)] = np.nanmin(data)

    denom = np.nanmax(data) - np.nanmin(data)
    if denom == 0:
        scaled = np.zeros_like(data)
    else:
        scaled = (data - np.nanmin(data)) / denom * 255

    image = Image.fromarray(scaled.astype(np.uint8), mode="L")
    image.save(output_png_path)

## APPLY PNG MASK

In [ ]:
def apply_mask_png(image_path: str, mask_path: str, output_path: str):
    image = Image.open(image_path).convert("L")
    mask = Image.open(mask_path).convert("L")

    image_array = np.array(image)
    mask_array = np.array(mask)

    if image_array.shape != mask_array.shape:
        print(f"Shape mismatch: {image_path}")
        return

    mask_binary = (mask_array > 0).astype(np.uint8)
    masked = np.where(mask_binary == 1, image_array, 0)

    Image.fromarray(masked.astype(np.uint8), mode="L").save(output_path)

## Bulk processing pipeline

In [ ]:
def create_output_folders(base_paths: dict, folder: str) -> dict:
    """Create all output subfolders for a given class/folder."""
    paths = {key: os.path.join(path, folder) for key, path in base_paths.items()}

    for path in paths.values():
        Path(path).mkdir(parents=True, exist_ok=True)

    return paths

## PyBDSF utilities

In [ ]:
def run_pybdsm(
    image_path: str,
    freq: float,
    beam: tuple,
    threshold_island: int,
    threshold_pixel: int,
    rms_box: tuple,
    atrous_do: bool,
    rms_map: bool,
    mean_map: str,
    flag_maxsize_bm: int,
):
    """Run PyBDSF on a FITS image."""
    return bdsf.process_image(
        image_path,
        frequency=freq,
        beam=beam,
        thresh_isl=threshold_island,
        thresh_pix=threshold_pixel,
        rms_box=rms_box,
        atrous_do=atrous_do,
        rms_map=rms_map,
        mean_map=mean_map,
        flag_maxsize_bm=flag_maxsize_bm,
    )


def export_spurious_mask(image, output_path: str):
    """Export full island mask (spurious included)."""
    image.export_image(
        img_type="island_mask",
        outfile=output_path,
        clobber=True,
        mask_dilation=1,
    )

## Island utilities

In [ ]:
def get_island_positions(image):
    """Get center positions of all detected islands."""
    island_positions = []

    for isl in image.islands:
        island_id = isl.island_id
        mask_active = isl.mask_active
        y_slice, x_slice = isl.bbox

        if mask_active.shape == image.ch0_arr.shape:
            mask_active = mask_active[y_slice, x_slice]

        coords = np.argwhere(~mask_active.astype(bool))
        if coords.size == 0:
            continue

        y0, x0 = np.median(coords, axis=0)
        xpos = x0 + x_slice.start
        ypos = y0 + y_slice.start

        island_positions.append((island_id, xpos, ypos))

    return island_positions


def find_central_island(image, island_positions):
    """Find island closest to image center."""
    image_center = np.array([image.ch0_arr.shape[1] / 2, image.ch0_arr.shape[0] / 2])

    distances = [
        (iid, np.linalg.norm(np.array([x, y]) - image_center))
        for iid, x, y in island_positions
    ]

    return min(distances, key=lambda x: x[1])[0]


def find_neighbors(island_positions, neighbor_pixel_dist):
    """Build neighbor dictionary for all islands."""
    neighbor_dict = {}

    for id1, x1, y1 in island_positions:
        neighbors = []

        for id2, x2, y2 in island_positions:
            if id1 == id2:
                continue

            dist = np.sqrt((x1 - x2) ** 2 + (y1 - y2) ** 2)

            if dist <= neighbor_pixel_dist:
                neighbors.append(id2)

        neighbor_dict[id1] = neighbors

    return neighbor_dict

## Mask utilities

In [ ]:
def export_non_spurious_mask(
    image,
    central_island_id,
    central_neighbors,
    output_path: str,
):
    """Export cleaned island mask keeping central source + neighbors."""
    original_pyrank = image.pyrank.copy()
    keep_mask = np.zeros_like(original_pyrank, dtype=bool)

    for isl in image.islands:
        if isl.island_id != central_island_id and isl.island_id not in central_neighbors:
            continue

        y_slice, x_slice = isl.bbox
        mask_active = isl.mask_active

        if mask_active.shape == image.ch0_arr.shape:
            mask_active = mask_active[y_slice, x_slice]

        keep_mask[y_slice, x_slice] |= (~mask_active).astype(bool)

    image.pyrank = np.where(keep_mask, original_pyrank, -1).astype(original_pyrank.dtype)

    image.export_image(
        img_type="island_mask",
        outfile=output_path,
        clobber=True,
        mask_dilation=1,
    )

    image.pyrank = original_pyrank

## FITS file processing

In [ ]:
def process_single_fits(
    image_path: str,
    stem: str,
    folder_paths: dict,
    config: dict,
):
    """Process one FITS file and generate masks/PNGs."""

    image = run_pybdsm(
        image_path=image_path,
        freq=config["freq"],
        beam=config["beam"],
        threshold_island=config["threshold_island"],
        threshold_pixel=config["threshold_pixel"],
        rms_box=config["rms_box"],
        atrous_do=config["atrous_do"],
        rms_map=config["rms_map"],
        mean_map=config["mean_map"],
        flag_maxsize_bm=config["flag_maxsize_bm"],
    )

    spurious_mask_fits = os.path.join(folder_paths["spurious_mask_fits"], stem + "_mask.fits")
    export_spurious_mask(image, spurious_mask_fits)

    island_positions = get_island_positions(image)
    if len(island_positions) == 0:
        return

    central_island_id = find_central_island(image, island_positions)

    neighbor_dict = find_neighbors(island_positions, config["neighbor_pixel_dist"])
    central_neighbors = neighbor_dict.get(central_island_id, [])

    no_spurious_mask_fits = os.path.join(folder_paths["no_spurious_mask_fits"], stem + "_mask.fits")
    export_non_spurious_mask(
        image=image,
        central_island_id=central_island_id,
        central_neighbors=central_neighbors,
        output_path=no_spurious_mask_fits,
    )

    raw_png_path = os.path.join(folder_paths["raw_png"], stem + ".png")
    fits_to_png(image_path, raw_png_path)

## Conversion utilities

In [ ]:
def convert_masks_to_png(mask_fits_folder: str, mask_png_folder: str):
    """Convert all FITS masks in a folder to PNG."""
    for mask_file in os.listdir(mask_fits_folder):
        if mask_file.endswith(".fits"):
            fits_to_png(
                os.path.join(mask_fits_folder, mask_file),
                os.path.join(mask_png_folder, mask_file.replace(".fits", ".png")),
            )


def apply_masks_to_folder(folder_paths: dict):
    """Apply spurious and non-spurious masks to raw PNGs."""
    for image_file in os.listdir(folder_paths["raw_png"]):
        if not image_file.endswith(".png"):
            continue

        stem = Path(image_file).stem
        raw_png_path = os.path.join(folder_paths["raw_png"], image_file)

        spurious_mask_png = os.path.join(folder_paths["spurious_mask_png"], stem + "_mask.png")
        no_spurious_mask_png = os.path.join(folder_paths["no_spurious_mask_png"], stem + "_mask.png")

        if os.path.exists(spurious_mask_png):
            apply_mask_png(
                raw_png_path,
                spurious_mask_png,
                os.path.join(folder_paths["spurious_masked_png"], stem + "_masked.png"),
            )

        if os.path.exists(no_spurious_mask_png):
            apply_mask_png(
                raw_png_path,
                no_spurious_mask_png,
                os.path.join(folder_paths["no_spurious_masked_png"], stem + "_masked.png"),
            )

## Folder processing

In [ ]:
def process_folder(
    folder: str,
    raw_fits_root: str,
    base_paths: dict,
    config: dict,
):
    """Process all FITS files in a single folder."""
    raw_fits_folder = os.path.join(raw_fits_root, folder)
    if not os.path.isdir(raw_fits_folder):
        return

    print(f"\n========== Processing folder: {folder} ==========")

    folder_paths = create_output_folders(base_paths, folder)

    for image_file in os.listdir(raw_fits_folder):
        if not image_file.endswith(".fits"):
            continue

        print(f"Processing: {image_file}")
        image_path = os.path.join(raw_fits_folder, image_file)
        stem = Path(image_file).stem

        try:
            process_single_fits(
                image_path=image_path,
                stem=stem,
                folder_paths=folder_paths,
                config=config,
            )
        except Exception as e:
            print(f"Error processing {image_file}: {e}")
            continue

    convert_masks_to_png(folder_paths["spurious_mask_fits"], folder_paths["spurious_mask_png"])
    convert_masks_to_png(folder_paths["no_spurious_mask_fits"], folder_paths["no_spurious_mask_png"])

    apply_masks_to_folder(folder_paths)

## Dataset pipeline

In [ ]:
def process_dataset(root_dataset_path: str):
    """Main dataset processing pipeline."""
    raw_fits_root = os.path.join(root_dataset_path, "raw_fits")

    base_paths = {
        "raw_png": os.path.join(root_dataset_path, "raw_png"),
        "spurious_mask_fits": os.path.join(root_dataset_path, "spurious_mask_fits"),
        "no_spurious_mask_fits": os.path.join(root_dataset_path, "no_spurious_mask_fits"),
        "spurious_mask_png": os.path.join(root_dataset_path, "spurious_mask_png"),
        "no_spurious_mask_png": os.path.join(root_dataset_path, "no_spurious_mask_png"),
        "spurious_masked_png": os.path.join(root_dataset_path, "spurious_masked_png"),
        "no_spurious_masked_png": os.path.join(root_dataset_path, "no_spurious_masked_png"),
    }

    config = {
        "beam": (5.4 / 3600, 5.4 / 3600, 0.0),
        "freq": 1.4e9,
        "threshold_island": 3,
        "threshold_pixel": 5,
        "rms_box": (25, 8),
        "atrous_do": True,
        "rms_map": False,
        "mean_map": "zero",
        "flag_maxsize_bm": 50,
        "neighbor_pixel_dist": 30,
    }

    for folder in os.listdir(raw_fits_root):
        process_folder(
            folder=folder,
            raw_fits_root=raw_fits_root,
            base_paths=base_paths,
            config=config,
        )

## Entry point

In [ ]:
def main():
    root_dataset_path = "/path/to/rgc/new_dataset/"
    process_dataset(root_dataset_path)


if __name__ == "__main__":
    main()